# Random Disclosure Parity Game: Probability Distributions & Payoff Table

## Game Definition

> *The Random Disclosure Parity Game is a two-player game involving coin-toss and a number choice and a total of four moves.*
>
> 1. **Move 1 (Player I):** Selects one of the two integers $\{1, 2\}$.
> 2. **Move 2 (Referee — coin toss):** The referee tosses a fair coin. If the outcome is **heads**, he informs Player II of Player I's choice; if **tails**, Player II receives no information.
> 3. **Move 3 (Player II):** Selects an integer from $\{3, 4\}$.
> 4. **Move 4 (Referee — random draw):** The referee selects one of $\{1, 2, 3\}$ with respective probabilities $\{0.4, 0.2, 0.4\}$.
>
> The numbers selected in moves 1, 3, and 4 are **summed**. If the sum is **even**, Player II pays Player I one dollar; if the sum is **odd**, Player I pays Player II one dollar.

### Payoff to Player I

$$
\text{Payoff}(i, j, r) = 
\begin{cases}
+1 & \text{if } i + j + r \text{ is even} \\
-1 & \text{if } i + j + r \text{ is odd}
\end{cases}
$$

where $i$ is Player I's choice, $j$ is Player II's choice, and $r$ is the referee's draw.

In [ ]:
import numpy as np
import pandas as pd
from itertools import product
import matplotlib.pyplot as plt

plt.style.use('dark_background')
plt.rcParams['font.family'] = 'monospace'
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.facecolor'] = '#1e1e2e'
plt.rcParams['figure.facecolor'] = '#11111b'
plt.rcParams['axes.edgecolor'] = '#6c7086'
plt.rcParams['axes.labelcolor'] = '#89b4fa'
plt.rcParams['xtick.color'] = '#a6adc8'
plt.rcParams['ytick.color'] = '#a6adc8'
plt.rcParams['text.color'] = '#cdd6f4'
plt.rcParams['grid.color'] = '#313244'
plt.rcParams['grid.alpha'] = 0.5

## 1. Information Sets and Player Strategies

### Player I

Player I moves first with no prior information. Their strategy is simply a choice from $\{1, 2\}$:

| Strategy | Action |
|----------|--------|
| $\sigma_I^1$ | Choose 1 |
| $\sigma_I^2$ | Choose 2 |

### Player II

Player II's **information sets** depend on the coin toss:

| Information Set | Condition | Player II knows |
|:---:|:---|:---|
| $H_1$ | Heads and Player I chose 1 | Player I chose 1 |
| $H_2$ | Heads and Player I chose 2 | Player I chose 2 |
| $T$   | Tails (regardless of Player I's choice) | Nothing about Player I's choice |

A **strategy** for Player II specifies an action $\in \{3, 4\}$ at each information set.
With 3 information sets and 2 actions each, Player II has $2^3 = 8$ pure strategies:

$$\sigma_{II} = (a_{H_1},\; a_{H_2},\; a_T) \in \{3,4\}^3$$

In [ ]:
PI_ACTIONS = [1, 2]
PII_ACTIONS = [3, 4]
REFEREE_OUTCOMES = [1, 2, 3]
REFEREE_PROBS = {1: 0.4, 2: 0.2, 3: 0.4}
COIN_PROB = 0.5  # P(heads) = P(tails) = 0.5

# Player I strategies: just the choice itself
pi_strategies = PI_ACTIONS  # [1, 2]

# Player II strategies: (action_at_H1, action_at_H2, action_at_T)
pii_strategies = list(product(PII_ACTIONS, PII_ACTIONS, PII_ACTIONS))

print("Player I strategies:")
for idx, s in enumerate(pi_strategies, 1):
    print(f"  σ_I^{idx}: choose {s}")

print(f"\nPlayer II strategies (action at H₁, H₂, T):")
for idx, (a, b, c) in enumerate(pii_strategies, 1):
    print(f"  σ_II^{idx}: ({a}, {b}, {c})  "
          f"→ play {a} if informed PI chose 1, "
          f"play {b} if informed PI chose 2, "
          f"play {c} if uninformed")

## 2. Probability Distributions over Sums

For a given strategy pair $(\sigma_I, \sigma_{II})$, the sum $S = i + j + r$ is a random variable whose distribution depends on the coin toss and the referee's draw.

Given Player I chooses $i$ and Player II uses strategy $(a_{H_1}, a_{H_2}, a_T)$:

$$
P(S = s) = \underbrace{\frac{1}{2} \cdot P(r = s - i - j_{\text{heads}})}_{\text{coin = heads}} \;+\; \underbrace{\frac{1}{2} \cdot P(r = s - i - a_T)}_{\text{coin = tails}}
$$

where $j_{\text{heads}} = a_{H_1}$ if $i = 1$, or $a_{H_2}$ if $i = 2$.

The possible sums range from $1 + 3 + 1 = 5$ to $2 + 4 + 3 = 9$.

In [ ]:
ALL_POSSIBLE_SUMS = list(range(5, 10))  # 5, 6, 7, 8, 9


def compute_distribution(pi_choice, pii_strategy):
    """
    Compute P(S = s) for each possible sum s, given Player I's choice
    and Player II's strategy (a_H1, a_H2, a_T).

    Returns a dict {sum: probability}.
    """
    a_h1, a_h2, a_t = pii_strategy

    # Player II's action when informed (heads)
    j_heads = a_h1 if pi_choice == 1 else a_h2
    # Player II's action when uninformed (tails)
    j_tails = a_t

    dist = {s: 0.0 for s in ALL_POSSIBLE_SUMS}

    for r, p_r in REFEREE_PROBS.items():
        # Heads branch
        s_heads = pi_choice + j_heads + r
        dist[s_heads] = dist.get(s_heads, 0.0) + COIN_PROB * p_r

        # Tails branch
        s_tails = pi_choice + j_tails + r
        dist[s_tails] = dist.get(s_tails, 0.0) + COIN_PROB * p_r

    # Remove zero-probability sums
    return {s: p for s, p in sorted(dist.items()) if p > 0}


# Compute distributions for all 16 strategy pairs
distributions = {}
for pi_s in pi_strategies:
    for pii_s in pii_strategies:
        distributions[(pi_s, pii_s)] = compute_distribution(pi_s, pii_s)

print(f"Computed probability distributions for {len(distributions)} strategy pairs.")

In [ ]:
def format_dist_table(distributions):
    """Build a DataFrame showing P(S=s) for each strategy pair."""
    rows = []
    for (pi_s, pii_s), dist in sorted(distributions.items()):
        row = {
            'PI': f'choose {pi_s}',
            'PII (H₁,H₂,T)': f'({pii_s[0]},{pii_s[1]},{pii_s[2]})',
        }
        for s in ALL_POSSIBLE_SUMS:
            p = dist.get(s, 0.0)
            row[f'P(S={s})'] = p if p > 0 else 0.0
        # Parity breakdown
        row['P(even)'] = sum(p for s, p in dist.items() if s % 2 == 0)
        row['P(odd)'] = sum(p for s, p in dist.items() if s % 2 != 0)
        rows.append(row)
    return pd.DataFrame(rows)


dist_df = format_dist_table(distributions)
dist_df.index = [f"({row['PI']}, {row['PII (H₁,H₂,T)']})" for _, row in dist_df.iterrows()]
dist_df.index.name = 'Strategy Pair'

print("Probability distributions over sums for all 16 strategy pairs:")
print("(Rows: Player I strategy × Player II strategy)\n")
display_cols = [c for c in dist_df.columns if c.startswith('P(')]
dist_df[display_cols].style.format('{:.2f}').background_gradient(
    cmap='YlOrRd', subset=[c for c in display_cols if 'S=' in c]
)

### Detailed Breakdown by Coin Outcome

For a deeper look, we decompose each distribution into its **heads** and **tails** components. This shows exactly how the disclosure mechanism affects the outcome probabilities.

In [ ]:
def detailed_breakdown(pi_choice, pii_strategy):
    """
    Show the probability tree for one strategy pair,
    broken down by coin outcome.
    """
    a_h1, a_h2, a_t = pii_strategy
    j_heads = a_h1 if pi_choice == 1 else a_h2
    j_tails = a_t

    print(f"  Player I chooses: {pi_choice}")
    print(f"  Player II strategy: (H₁→{a_h1}, H₂→{a_h2}, T→{a_t})")
    print()

    for branch, j, label in [("Heads (p=0.5)", j_heads, "informed"),
                              ("Tails (p=0.5)", j_tails, "uninformed")]:
        print(f"  ┌─ {branch}: PII plays {j} ({label})")
        for r, p_r in REFEREE_PROBS.items():
            s = pi_choice + j + r
            parity = "even" if s % 2 == 0 else "odd"
            payoff = +1 if s % 2 == 0 else -1
            joint_p = COIN_PROB * p_r
            print(f"  │   r={r} (p={p_r}): sum = {pi_choice}+{j}+{r} = {s} "
                  f"({parity}) → payoff {payoff:+d}  [joint prob = {joint_p:.2f}]")
        print(f"  └")
    print()


print("═" * 72)
print("  DETAILED BREAKDOWN: Two representative strategy pairs")
print("═" * 72)

print(f"\n── Example 1: PI chooses 1, PII strategy (3, 3, 3) ──\n")
detailed_breakdown(1, (3, 3, 3))

print(f"── Example 2: PI chooses 1, PII strategy (4, 3, 3) ──\n")
detailed_breakdown(1, (4, 3, 3))

print(f"── Example 3: PI chooses 2, PII strategy (3, 4, 3) ──\n")
detailed_breakdown(2, (3, 4, 3))

## 3. Expected Payoff Computation

The expected payoff to Player I for strategy pair $(\sigma_I, \sigma_{II})$ is:

$$
E[\text{payoff}] = P(S \text{ even}) \cdot (+1) + P(S \text{ odd}) \cdot (-1) = P(\text{even}) - P(\text{odd})
$$

### Key structural observation

The parity of $S = i + j + r$ depends only on the parity of $i + j$ (since $r$ is then drawn):

- If $i + j$ is **even**: $P(S \text{ even}) = P(r = 2) = 0.2$, so $E[\text{payoff}] = 0.2 - 0.8 = -0.6$
- If $i + j$ is **odd**: $P(S \text{ even}) = P(r = 1) + P(r = 3) = 0.8$, so $E[\text{payoff}] = 0.8 - 0.2 = +0.6$

The conditional payoffs for each $(i, j)$ combination are:

| $i$ | $j$ | $i + j$ | Parity | $E[\text{payoff} \mid i, j]$ |
|:---:|:---:|:-------:|:------:|:---:|
| 1 | 3 | 4 | even | $-0.6$ |
| 1 | 4 | 5 | odd  | $+0.6$ |
| 2 | 3 | 5 | odd  | $+0.6$ |
| 2 | 4 | 6 | even | $-0.6$ |

Player I wants $i + j$ to be **odd** (different parities), while Player II wants $i + j$ to be **even** (same parities). The disclosure mechanism is what makes this non-trivial — when the coin is heads, Player II can **match** Player I's parity.

In [ ]:
def expected_payoff(dist):
    """E[payoff to PI] = P(even) - P(odd)."""
    return sum(p * (1 if s % 2 == 0 else -1) for s, p in dist.items())


def build_payoff_table(distributions):
    """Build the 2×8 payoff matrix."""
    payoffs = {}
    for (pi_s, pii_s), dist in distributions.items():
        payoffs[(pi_s, pii_s)] = expected_payoff(dist)

    col_labels = [f"({a},{b},{c})" for a, b, c in pii_strategies]
    row_labels = [f"PI: {s}" for s in pi_strategies]

    matrix = np.array([
        [payoffs[(pi_s, pii_s)] for pii_s in pii_strategies]
        for pi_s in pi_strategies
    ])

    return pd.DataFrame(matrix, index=row_labels, columns=col_labels), matrix


payoff_df, payoff_matrix = build_payoff_table(distributions)
payoff_df.columns.name = 'PII: (H₁,H₂,T)'
payoff_df.index.name = 'Player I'

print("═" * 72)
print("  PAYOFF TABLE  (expected dollars from Player II to Player I)")
print("═" * 72)
print()
payoff_df.style.format('{:+.1f}').background_gradient(
    cmap='RdYlGn', vmin=-0.6, vmax=0.6
)

In [ ]:
print("Payoff table (plain text):\n")

header = "PII strategy (H₁,H₂,T):"
col_strs = [f"({a},{b},{c})" for a, b, c in pii_strategies]
print(f"  {'':>8}  │ " + " │ ".join(f"{c:>7}" for c in col_strs) + " │")
print(f"  {'─'*8}──┼" + "─┼─".join('─' * 7 for _ in col_strs) + "─┤")

for pi_idx, pi_s in enumerate(pi_strategies):
    vals = [payoff_matrix[pi_idx, pii_idx] for pii_idx in range(len(pii_strategies))]
    val_strs = [f"{v:+.1f}" for v in vals]
    print(f"  PI = {pi_s}   │ " + " │ ".join(f"{v:>7}" for v in val_strs) + " │")

print()
print("Positive values favor Player I; negative values favor Player II.")

## 4. Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

colors = ['#f38ba8', '#89b4fa']
bar_width = 0.35

for ax_idx, pi_s in enumerate(pi_strategies):
    ax = axes[ax_idx]
    x = np.arange(len(pii_strategies))

    vals = [payoff_matrix[ax_idx, j] for j in range(len(pii_strategies))]
    bar_colors = ['#a6e3a1' if v > 0 else '#f38ba8' if v < 0 else '#a6adc8' for v in vals]

    bars = ax.bar(x, vals, color=bar_colors, edgecolor='#45475a', linewidth=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels([f"({a},{b},{c})" for a, b, c in pii_strategies],
                        rotation=45, ha='right', fontsize=9)
    ax.set_xlabel('Player II strategy (H₁, H₂, T)')
    ax.set_ylabel('Expected payoff to Player I ($)')
    ax.set_title(f'Player I chooses {pi_s}', fontsize=13, fontweight='bold')
    ax.axhline(y=0, color='#585b70', linewidth=1, linestyle='--')
    ax.set_ylim(-0.8, 0.8)
    ax.grid(axis='y', alpha=0.3)

    for bar, v in zip(bars, vals):
        y_pos = bar.get_height() + 0.03 if v >= 0 else bar.get_height() - 0.07
        ax.text(bar.get_x() + bar.get_width() / 2, y_pos, f'{v:+.1f}',
                ha='center', va='bottom' if v >= 0 else 'top', fontsize=9,
                fontweight='bold')

fig.suptitle('Payoff Table: Random Disclosure Parity Game',
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 8), sharey=True)

sum_labels = [str(s) for s in ALL_POSSIBLE_SUMS]

for pi_idx, pi_s in enumerate(pi_strategies):
    for pii_idx, pii_s in enumerate(pii_strategies):
        row = pi_idx
        col = pii_idx % 4 if pi_idx == 0 else pii_idx % 4
        if pii_idx >= 4 and pi_idx == 0:
            continue
        if pii_idx < 4 and pi_idx == 1:
            continue

        ax = axes[row][col]
        dist = distributions[(pi_s, pii_s)]

        probs = [dist.get(s, 0.0) for s in ALL_POSSIBLE_SUMS]
        bar_colors = ['#a6e3a1' if s % 2 == 0 else '#f38ba8' for s in ALL_POSSIBLE_SUMS]

        ax.bar(sum_labels, probs, color=bar_colors, edgecolor='#45475a', linewidth=0.8)
        ax.set_title(f'PI={pi_s}, PII=({pii_s[0]},{pii_s[1]},{pii_s[2]})',
                     fontsize=9, fontweight='bold')
        ax.set_ylim(0, 0.5)
        ax.set_xlabel('Sum', fontsize=8)
        if col == 0:
            ax.set_ylabel('Probability', fontsize=8)

        ep = expected_payoff(dist)
        ax.text(0.95, 0.90, f'E={ep:+.1f}', transform=ax.transAxes,
                ha='right', va='top', fontsize=9, fontweight='bold',
                color='#a6e3a1' if ep > 0 else '#f38ba8' if ep < 0 else '#a6adc8',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='#313244', alpha=0.8))

fig.suptitle('Probability Distributions over Sums (green = even, red = odd)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

Let's also show all 16 distributions in a single comprehensive grid.

In [ ]:
fig, axes = plt.subplots(2, 8, figsize=(22, 6), sharey=True)

for pi_idx, pi_s in enumerate(pi_strategies):
    for pii_idx, pii_s in enumerate(pii_strategies):
        ax = axes[pi_idx][pii_idx]
        dist = distributions[(pi_s, pii_s)]

        probs = [dist.get(s, 0.0) for s in ALL_POSSIBLE_SUMS]
        bar_colors = ['#a6e3a1' if s % 2 == 0 else '#f38ba8' for s in ALL_POSSIBLE_SUMS]

        ax.bar(sum_labels, probs, color=bar_colors, edgecolor='#45475a', linewidth=0.5)
        ax.set_ylim(0, 0.5)

        if pi_idx == 0:
            ax.set_title(f'PII=({pii_s[0]},{pii_s[1]},{pii_s[2]})', fontsize=8)
        if pii_idx == 0:
            ax.set_ylabel(f'PI={pi_s}', fontsize=10, fontweight='bold')
        ax.tick_params(labelsize=7)

        ep = expected_payoff(dist)
        ax.text(0.50, 0.92, f'E={ep:+.1f}', transform=ax.transAxes,
                ha='center', va='top', fontsize=8, fontweight='bold',
                color='#a6e3a1' if ep > 0 else '#f38ba8' if ep < 0 else '#a6adc8',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='#313244', alpha=0.8))

fig.suptitle('All 16 Probability Distributions  (green = even sum → PI wins, red = odd sum → PII wins)',
             fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 5. Analysis of the Payoff Table

### Dominance and structure

In [ ]:
print("═" * 72)
print("  PAYOFF TABLE ANALYSIS")
print("═" * 72)

print("\n── Payoff matrix (rows = PI, columns = PII) ──\n")
print(payoff_df.to_string())

# Check for dominated strategies
print("\n── Dominated strategy analysis ──\n")

# Player I: check if either row dominates the other
row1, row2 = payoff_matrix[0], payoff_matrix[1]
if np.all(row1 >= row2) and np.any(row1 > row2):
    print("  Player I: 'choose 1' dominates 'choose 2'")
elif np.all(row2 >= row1) and np.any(row2 > row1):
    print("  Player I: 'choose 2' dominates 'choose 1'")
else:
    print("  Player I: neither strategy dominates the other")

# Player II: check pairwise dominance (PII is minimizer)
n_pii = len(pii_strategies)
dominated = set()
for i in range(n_pii):
    for j in range(n_pii):
        if i == j:
            continue
        col_i = payoff_matrix[:, i]
        col_j = payoff_matrix[:, j]
        # Strategy j dominates strategy i for PII (minimizer)
        # if col_j <= col_i everywhere and strictly less somewhere
        if np.all(col_j <= col_i) and np.any(col_j < col_i):
            dominated.add(i)

if dominated:
    print(f"  Player II: dominated strategies (for the minimizer):")
    for idx in sorted(dominated):
        s = pii_strategies[idx]
        # Find which strategy dominates it
        dominators = []
        for j in range(n_pii):
            if j == idx:
                continue
            col_j = payoff_matrix[:, j]
            col_i = payoff_matrix[:, idx]
            if np.all(col_j <= col_i) and np.any(col_j < col_i):
                dominators.append(pii_strategies[j])
        dom_str = ', '.join(str(d) for d in dominators)
        print(f"    ({s[0]},{s[1]},{s[2]}) is dominated by: {dom_str}")
else:
    print("  Player II: no dominated strategies found")

# Maximin / minimax
print("\n── Security levels ──\n")

# PI's maximin: for each PI strategy, worst-case over PII strategies
pi_security = [np.min(payoff_matrix[i, :]) for i in range(len(pi_strategies))]
for i, s in enumerate(pi_strategies):
    print(f"  Player I 'choose {s}': worst-case payoff = {pi_security[i]:+.1f}")
maximin = max(pi_security)
print(f"  → Player I's maximin value = {maximin:+.1f}")

print()

# PII's minimax: for each PII strategy, worst-case (highest payoff to PI) over PI strategies
pii_security = [np.max(payoff_matrix[:, j]) for j in range(n_pii)]
for j, s in enumerate(pii_strategies):
    print(f"  Player II ({s[0]},{s[1]},{s[2]}): worst-case = {pii_security[j]:+.1f}")
minimax = min(pii_security)
print(f"  → Player II's minimax value = {minimax:+.1f}")

print(f"\n── Summary ──\n")
print(f"  Maximin (PI)  = {maximin:+.1f}")
print(f"  Minimax (PII) = {minimax:+.1f}")
if abs(maximin - minimax) < 1e-9:
    print(f"  Maximin = Minimax → pure strategy saddle point exists!")
    print(f"  Game value = {maximin:+.1f}")
else:
    print(f"  Maximin ≠ Minimax → no pure strategy saddle point.")
    print(f"  The game value (in mixed strategies) lies in [{maximin:+.1f}, {minimax:+.1f}].")

## 6. Summary

We have computed:

1. **All 16 probability distributions** over the sum $S \in \{5,6,7,8,9\}$ for every pair of pure strategies.

2. **The $2 \times 8$ payoff table** giving the expected payoff to Player I for each strategy pair.

3. **Structural insight**: the expected payoff for any scenario where Player I plays $i$ and Player II plays $j$ depends only on the parity of $i + j$:
   - $i + j$ even $\Rightarrow E = -0.6$ (favors Player II)
   - $i + j$ odd $\Rightarrow E = +0.6$ (favors Player I)

4. **The disclosure mechanism** is the key strategic element: when the coin is heads, Player II can observe Player I's choice and **match parity** to guarantee a favorable outcome in that branch. This asymmetry means the coin toss genuinely affects the strategic landscape.